# Fast Style Transfer

Single-style optimization baseline plus hyperparameter sweeps via reusable utilities.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import tensorflow as tf
from models.fast_style_transfer import (
    ExperimentConfig,
    StyleTransfer,
    load_and_preprocess,
    plot_image_mosaic,
    plot_loss_mosaic,
    plot_training_curves,
    run_experiments,
    show_style_transfer_triplet,
    train_style_transfer,
)

if tf.config.list_physical_devices("GPU"):
    print("TensorFlow will use GPU acceleration!")
else:
    print("Running on CPU.")


In [ ]:
style_image = load_and_preprocess("data/van_gogh_style.jpg")
content_image = load_and_preprocess("data/example.jpg")


In [ ]:
BASELINE_MODEL = StyleTransfer(
    style_image=style_image,
    content_image=content_image,
    content_weight=10,
    style_weight=1e3,
    tvl_weight=10,
)

BASELINE_EPOCHS = 1000
baseline_history = train_style_transfer(
    BASELINE_MODEL, epochs=BASELINE_EPOCHS, lr=1e-2, verbose_every=100
)


In [ ]:
plot_training_curves(baseline_history)

In [ ]:
baseline_output = tf.squeeze(BASELINE_MODEL.call(), axis=0).numpy()
show_style_transfer_triplet(
    style_image.numpy(), content_image.numpy(), baseline_output
)


In [ ]:
experiment_configs = [
    ExperimentConfig("A", content_weight=1e4, style_weight=1e0),
    ExperimentConfig("B", content_weight=1e0, style_weight=1e4),
    ExperimentConfig("C", content_weight=1e1, style_weight=1e1),
    ExperimentConfig("D", content_weight=1e1, style_weight=1e1, lr=10),
    ExperimentConfig("E", content_weight=1e-5, style_weight=1e1),
    ExperimentConfig("F", content_weight=1e1, style_weight=1e-5),
    ExperimentConfig("G", content_weight=1e0, style_weight=1e2),
]

experiments = run_experiments(
    style_image, content_image, experiment_configs, verbose_every=200
)


In [ ]:
plot_loss_mosaic(experiments)

In [ ]:
plot_image_mosaic(
    experiments, style_image.numpy(), content_image.numpy()
)
